In [2]:
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
OUT = Path("eda_artifacts/pseudo_gt")
OUT.mkdir(parents=True, exist_ok=True)

TARGET_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)
DATE_AGREEMENT_DAYS = 90

WINDOW_START = date(TARGET_YEARS[0], 1, 1).toordinal()
WINDOW_END   = date(TARGET_YEARS[-1] + 1, 1, 1).toordinal()
EPOCH_RADD   = date(2014, 12, 31).toordinal()
EPOCH_GLADS2 = date(2019, 1, 1).toordinal()


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile):
    """Reference grid for the tile. Tries S2, GLAD-L, then RADD."""
    s2_dir = ROOT / f"sentinel-2/train/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best
    gl = sorted((ROOT / "labels/train/gladl").glob(f"gladl_{tile}_alert*.tif"))
    if gl:
        with rasterio.open(gl[0]) as r:
            return r.transform, r.crs, r.shape
    p = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if p.exists():
        with rasterio.open(p) as r:
            return r.transform, r.crs, r.shape
    return None


def reproj(path, ref, dtype):
    out = np.zeros(ref[2], dtype=dtype)
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), out,
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=ref[0], dst_crs=ref[1],
                  resampling=Resampling.nearest)
    return out


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape
    h_tgt, w_tgt = target_shape
    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    return arr[row_idx[:, None], col_idx[None, :]]


# ============================================================
# LABEL LOADERS
# ============================================================
def get_radd(tile, ref):
    p = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if not p.exists(): return None, None, False
    raw = reproj(p, ref, dtype=np.int32)
    days = raw % 10000
    date_ord = np.where(raw > 0, days + EPOCH_RADD, 0)
    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (raw > 0) & in_window
    return mask, np.where(mask, date_ord, 0), True


def get_glads2(tile, ref):
    ap = ROOT / f"labels/train/glads2/glads2_{tile}_alert.tif"
    dp = ROOT / f"labels/train/glads2/glads2_{tile}_alertDate.tif"
    if not (ap.exists() and dp.exists()): return None, None, False
    alert = reproj(ap, ref, dtype=np.uint8)
    days  = reproj(dp, ref, dtype=np.uint16)
    date_ord = np.where(alert > 0, days.astype(np.int64) + EPOCH_GLADS2, 0)
    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (alert >= 2) & in_window
    return mask, np.where(mask, date_ord, 0), True


def get_gladl(tile, ref):
    any_file = False
    mask = np.zeros(ref[2], dtype=bool)
    date_ord = np.zeros(ref[2], dtype=np.int64)
    for y in TARGET_YEARS:
        ap = ROOT / f"labels/train/gladl/gladl_{tile}_alert{y%100:02d}.tif"
        dp = ROOT / f"labels/train/gladl/gladl_{tile}_alertDate{y%100:02d}.tif"
        if not (ap.exists() and dp.exists()): continue
        any_file = True
        alert = reproj(ap, ref, dtype=np.uint8)
        doy = reproj(dp, ref, dtype=np.uint16)
        year_start = date(y, 1, 1).toordinal()
        year_date = np.where(alert > 0, doy.astype(np.int64) + year_start - 1, 0)
        year_mask = alert > 0
        better = year_mask & ((date_ord == 0) | ((year_date > 0) & (year_date < date_ord)))
        date_ord = np.where(better, year_date, date_ord)
        mask = mask | year_mask
    if not any_file: return None, None, False
    return mask, date_ord, True


# ============================================================
# BUILDER
# ============================================================
def build_pgt(tile):
    ref = get_ref(tile)
    if ref is None: return None
    native_shape = ref[2]
    
    radd_mask, radd_date, has_radd = get_radd(tile, ref)
    gs2_mask,  gs2_date,  has_gs2  = get_glads2(tile, ref)
    gl_mask,   gl_date,   has_gl   = get_gladl(tile, ref)
    
    if radd_mask is None:
        radd_mask = np.zeros(native_shape, dtype=bool)
        radd_date = np.zeros(native_shape, dtype=np.int64)
    if gs2_mask is None:
        gs2_mask = np.zeros(native_shape, dtype=bool)
        gs2_date = np.zeros(native_shape, dtype=np.int64)
    if gl_mask is None:
        gl_mask = np.zeros(native_shape, dtype=bool)
        gl_date = np.zeros(native_shape, dtype=np.int64)
    
    votes = radd_mask.astype(np.int8) + gs2_mask.astype(np.int8) + gl_mask.astype(np.int8)
    n_sources = sum([has_radd, has_gs2, has_gl])
    
    # Confidence: 0.4 per vote
    confidence = votes.astype(np.float32) * 0.4
    
    # Temporal agreement: +0.15 per pair within 90 days
    def pair_bonus(m1, d1, m2, d2):
        both = m1 & m2
        if not both.any():
            return np.zeros(native_shape, dtype=np.float32)
        diff = np.abs(d1.astype(np.int64) - d2.astype(np.int64))
        close = both & (diff <= DATE_AGREEMENT_DAYS) & (d1 > 0) & (d2 > 0)
        return np.where(close, 0.15, 0.0).astype(np.float32)
    
    confidence += pair_bonus(radd_mask, radd_date, gs2_mask, gs2_date)
    confidence += pair_bonus(radd_mask, radd_date, gl_mask,  gl_date)
    confidence += pair_bonus(gs2_mask,  gs2_date,  gl_mask,  gl_date)
    confidence = np.clip(confidence, 0.0, 1.0)
    
    # Labels: 2+ sources agree = positive, all silent = negative, else NaN
    label = np.full(native_shape, np.nan, dtype=np.float32)
    if n_sources >= 2:
        label[votes >= 2] = 1.0
        label[votes == 0] = 0.0
    elif n_sources == 1:
        label[votes == 0] = 0.0
    
    confidence = np.where(np.isnan(label), 0.0, confidence)
    confidence = np.where(label == 0.0, 1.0, confidence)
    
    # Event year + month (earliest across sources)
    event_month = np.zeros(native_shape, dtype=np.int8)
    event_year = np.zeros(native_shape, dtype=np.int16)
    dates_stack = np.stack([radd_date, gs2_date, gl_date])
    masked = np.where(dates_stack > 0, dates_stack, np.iinfo(np.int64).max)
    earliest = masked.min(axis=0)
    has_date = earliest < np.iinfo(np.int64).max
    if has_date.any():
        for ord_val in np.unique(earliest[has_date]):
            m = (earliest == ord_val) & has_date & (label == 1.0)
            if m.any():
                d = date.fromordinal(int(ord_val))
                event_month[m] = d.month
                event_year[m] = d.year
    
    # Resize everything to (1000, 1000)
    label_r       = resize_nearest(label, TARGET_SHAPE)
    confidence_r  = resize_nearest(confidence, TARGET_SHAPE)
    event_month_r = resize_nearest(event_month, TARGET_SHAPE)
    event_year_r  = resize_nearest(event_year, TARGET_SHAPE)
    
    pos = label_r == 1.0
    total = label_r.size
    
    # Per-year positive counts
    per_year = {}
    for y in TARGET_YEARS:
        per_year[f"pos_y{y}"] = int(((event_year_r == y) & pos).sum())
    
    return {
        "label": label_r,
        "confidence": confidence_r,
        "event_month": event_month_r,
        "event_year": event_year_r,
        "native_shape": f"{native_shape[0]}x{native_shape[1]}",
        "n_sources": n_sources,
        "has": {"radd": has_radd, "gs2": has_gs2, "gl": has_gl},
        "pos_frac": float(pos.sum() / total),
        "neg_frac": float((label_r == 0.0).sum() / total),
        "ign_frac": float(np.isnan(label_r).sum() / total),
        "pos_mean_conf": float(confidence_r[pos].mean()) if pos.any() else 0.0,
        "strong_pos": int((pos & (confidence_r >= 0.95)).sum()),
        "medium_pos": int((pos & (confidence_r >= 0.80) & (confidence_r < 0.95)).sum()),
        "per_year": per_year,
    }


# ============================================================
# RUN
# ============================================================
train_tiles = sorted([p.name.replace("__s2_l2a", "")
                      for p in (ROOT / "sentinel-2/train").iterdir()])

stats = []
for tile in tqdm(train_tiles, desc=f"pseudo-GT @ {TARGET_SHAPE}"):
    try:
        r = build_pgt(tile)
        if r is None:
            print(f"  {tile}: skipped")
            continue
        np.savez_compressed(OUT / f"{tile}.npz",
                            label=r["label"],
                            confidence=r["confidence"],
                            event_month=r["event_month"],
                            event_year=r["event_year"])
        row = {
            "tile": tile,
            "native_shape": r["native_shape"],
            "n_sources": r["n_sources"],
            "has_radd": r["has"]["radd"],
            "has_gs2": r["has"]["gs2"],
            "has_gl": r["has"]["gl"],
            "pos_frac": r["pos_frac"],
            "neg_frac": r["neg_frac"],
            "ign_frac": r["ign_frac"],
            "pos_mean_conf": r["pos_mean_conf"],
            "strong_pos": r["strong_pos"],
            "medium_pos": r["medium_pos"],
        }
        row.update(r["per_year"])
        stats.append(row)
    except Exception as e:
        print(f"  {tile}: ERROR {e}")

df = pd.DataFrame(stats)
df.to_csv(OUT / "stats.csv", index=False)

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 30)
print("\n" + df.to_string(index=False))
print(f"\nTotal tiles: {len(df)}")
print(f"Mean pos_frac: {df.pos_frac.mean():.3%}")
print(f"Total strong pos: {df.strong_pos.sum():,}")
print(f"Total medium pos: {df.medium_pos.sum():,}")
print(f"\nPositives per year:")
for y in TARGET_YEARS:
    print(f"  {y}: {df[f'pos_y{y}'].sum():,}")

pseudo-GT @ (1000, 1000):   0%|          | 0/16 [00:00<?, ?it/s]


     tile native_shape  n_sources  has_radd  has_gs2  has_gl  pos_frac  neg_frac  ign_frac  pos_mean_conf  strong_pos  medium_pos  pos_y2020  pos_y2021  pos_y2022  pos_y2023  pos_y2024  pos_y2025
18NWG_6_6    1002x1002          3      True     True    True  0.365629  0.550460  0.083911       0.931223      280149       85480     108504     107603      57758      33338      37755      20671
18NWH_1_4    1002x1002          3      True     True    True  0.025868  0.950983  0.023149       0.904722       16736        9132      13654       5199       2859        823       2777        556
18NWJ_8_9    1002x1002          3      True     True    True  0.058639  0.892211  0.049150       0.903827       37815       20824      26172      11559      10718        483       4905       4802
18NWM_9_4      362x362          3      True     True    True  0.038479  0.913840  0.047681       0.927377       30026        8453       9048       9846       6144       5441       4224       3776
18NXH_6_8    1002x1

In [3]:
from pathlib import Path
ROOT = Path("data/makeathon-challenge/aef-embeddings/test")
for p in sorted(ROOT.glob("*.tiff")):
    print(p.name)

18NVJ_1_6_2020.tiff
18NVJ_1_6_2021.tiff
18NVJ_1_6_2022.tiff
18NVJ_1_6_2023.tiff
18NVJ_1_6_2024.tiff
18NVJ_1_6_2025.tiff
18NYH_2_1_2020.tiff
18NYH_2_1_2021.tiff
18NYH_2_1_2022.tiff
18NYH_2_1_2023.tiff
18NYH_2_1_2024.tiff
18NYH_2_1_2025.tiff
33NTE_5_1_2020.tiff
33NTE_5_1_2021.tiff
33NTE_5_1_2022.tiff
33NTE_5_1_2023.tiff
33NTE_5_1_2024.tiff
33NTE_5_1_2025.tiff
47QMA_6_2_2020.tiff
47QMA_6_2_2021.tiff
47QMA_6_2_2022.tiff
47QMA_6_2_2023.tiff
47QMA_6_2_2024.tiff
47QMA_6_2_2025.tiff
48PWA_0_6_2020.tiff
48PWA_0_6_2021.tiff
48PWA_0_6_2022.tiff
48PWA_0_6_2023.tiff
48PWA_0_6_2024.tiff
48PWA_0_6_2025.tiff


In [10]:
"""
Full pipeline: all-year AEF features → train LightGBM → predict test → GeoJSON submission
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import lightgbm as lgb
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import sys, json

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt")
MODEL_DIR = Path("eda_artifacts/model"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR = Path("submission"); SUB_DIR.mkdir(parents=True, exist_ok=True)
TILE_PRED_DIR = Path("eda_artifacts/predictions"); TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]   # all 6 years
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)    # 384 + 5 = 389

NEG_RATIO = 5
N_BOOST_ROUND = 400
CV_HOLDOUT = "18NXH_6_8"   # held-out validation tile


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    """Reference grid for tile (tries S2, then AEF)."""
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best
    # Fallback: AEF
    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape
    return None


def load_aef_to_ref(tile, year, split, ref):
    """Load AEF, reproject all 64 bands to reference grid."""
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"
    if not p.exists(): return None
    
    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)
        for b in range(N_AEF_BANDS):
            reproject(raw[b], aef[b],
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=ref[0], dst_crs=ref[1],
                      resampling=Resampling.bilinear)
    return aef


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape
    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]
    return arr[:, row_idx[:, None], col_idx[None, :]]


def extract_features(tile, split):
    """
    Returns features (N_pixels, N_FEATURES) and reference grid for tile.
    Features: 6×64 AEF values + 5 year-over-year cosine similarities.
    """
    ref = get_ref(tile, split)
    if ref is None: return None, None
    
    # Load all years
    aef_stack = []
    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)
        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None
        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)
    
    # Year-over-year cosine similarities
    changes = []
    for i in range(len(YEARS) - 1):
        a, b = aef_stack[i], aef_stack[i + 1]
        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b
        cos_sim = np.where(denom > 1e-6, dot / (denom + 1e-9), 0.0).astype(np.float32)
        changes.append(cos_sim)
    
    # Stack all features: shape (N_FEATURES, H, W) → (N_pixels, N_FEATURES)
    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0
    )
    features = all_feats.reshape(N_FEATURES, -1).T
    return features, ref


def feature_names():
    names = []
    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")
    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i+1]}")
    return names


# ============================================================
# STEP 1 — Extract features for training
# ============================================================
print(f"=== Step 1: Extract features (target = {N_FEATURES} per pixel) ===")
train_tiles = sorted([p.name.replace("__s2_l2a", "")
                      for p in (ROOT / "sentinel-2/train").iterdir()])

all_feats, all_labels, all_weights, all_tiles = [], [], [], []
for tile in tqdm(train_tiles, desc="train features"):
    pgt_path = PGT_DIR / f"{tile}.npz"
    if not pgt_path.exists(): continue
    
    feats, ref = extract_features(tile, "train")
    if feats is None: continue
    
    pgt = np.load(pgt_path)
    label = pgt["label"].flatten()
    conf = pgt["confidence"].flatten()
    
    # Keep only labeled pixels
    valid = ~np.isnan(label)
    all_feats.append(feats[valid])
    all_labels.append(label[valid])
    all_weights.append(conf[valid])
    all_tiles.append(np.full(int(valid.sum()), tile))

X = np.concatenate(all_feats, axis=0)
y = np.concatenate(all_labels).astype(np.float32)
w = np.concatenate(all_weights).astype(np.float32)
tile_ids = np.concatenate(all_tiles)

print(f"\nTotal labeled pixels: {len(y):,}")
print(f"Positives: {int((y == 1).sum()):,} ({(y == 1).mean():.2%})")
print(f"Negatives: {int((y == 0).sum()):,}")
print(f"Feature matrix: {X.shape}, dtype={X.dtype}")
print(f"Memory: {X.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 2 — Subsample negatives
# ============================================================
print(f"\n=== Step 2: Subsample negatives {NEG_RATIO}x ===")
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_RATIO)
rng = np.random.default_rng(42)
neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)
keep = np.concatenate([pos_idx, neg_sampled])
rng.shuffle(keep)

X_kept = X[keep]
y_kept = y[keep]
w_kept = w[keep]
tiles_kept = tile_ids[keep]

print(f"After subsampling: {len(y_kept):,} rows ({(y_kept == 1).mean():.1%} positive)")
print(f"Memory: {X_kept.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 3 — Held-out validation for threshold
# ============================================================
val_mask = tiles_kept == CV_HOLDOUT
tr_mask = ~val_mask
X_tr = X_kept[tr_mask]; y_tr = y_kept[tr_mask]; w_tr = w_kept[tr_mask]
X_val = X_kept[val_mask]; y_val = y_kept[val_mask]

print(f"\nValidation tile: {CV_HOLDOUT}")
print(f"  Train rows: {len(y_tr):,}, Val rows: {len(y_val):,}")


# ============================================================
# STEP 4 — Train LightGBM
# ============================================================
print("\n=== Step 4: Train LightGBM ===")
names = feature_names()

lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr, feature_name=names)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
}

model = lgb.train(
    params, lgb_train,
    num_boost_round=N_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

model.save_model(str(MODEL_DIR / "lgbm_allyears.txt"))


# ============================================================
# STEP 5 — Tune threshold
# ============================================================
print("\n=== Step 5: Tune threshold ===")
val_pred = model.predict(X_val)
best_thr, best_f1 = 0.5, 0
for thr in np.linspace(0.1, 0.9, 17):
    f1 = f1_score(y_val, val_pred > thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr
print(f"Best threshold: {best_thr:.2f} (F1={best_f1:.3f})")


# ============================================================
# STEP 6 — Predict test tiles
# ============================================================
print("\n=== Step 6: Predict test tiles ===")
test_tiles = sorted([p.name.replace("__s2_l2a", "")
                     for p in (ROOT / "sentinel-2/test").iterdir()])
print(f"Test tiles: {test_tiles}")

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue
    
    pred = model.predict(feats)
    binary = (pred > best_thr).astype(np.uint8).reshape(TARGET_SHAPE)
    
    # Resize binary to native resolution for georeferencing
    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE
    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)
    binary_native = binary[row_idx[:, None], col_idx[None, :]]
    
    # Save GeoTIFF
    tile_path = TILE_PRED_DIR / f"{tile}_pred.tif"
    profile = {
        "driver": "GTiff", "height": h_native, "width": w_native,
        "count": 1, "dtype": "uint8",
        "crs": ref[1], "transform": ref[0], "nodata": 0,
    }
    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)
    
    # Convert to GeoJSON (drops polygons < 0.5 ha inside submission_utils)
    geojson = raster_to_geojson(str(tile_path), output_path=None)
    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile
    all_features_geojson.extend(geojson["features"])
    print(f"  {tile}: {int(binary_native.sum()):,} positive pixels, "
          f"{len(geojson['features'])} polygons")


# ============================================================
# STEP 7 — Save submission
# ============================================================
submission = {"type": "FeatureCollection", "features": all_features_geojson}

sub_path = SUB_DIR / "submission.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\n=== Submission saved: {sub_path} ===")
print(f"Total polygons: {len(all_features_geojson)}")


# ============================================================
# STEP 8 — Feature importance (top 15)
# ============================================================
print("\n=== Feature importance (top 15) ===")
importance = pd.DataFrame({
    "feature": names,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
print(importance.head(15).to_string(index=False))

# Also aggregate by year to see which year's AEF matters most
print("\n=== Importance by year ===")
by_year = {y: 0 for y in YEARS}
change_total = 0
for _, row in importance.iterrows():
    feat = row["feature"]
    if feat.startswith("aef_"):
        year = int(feat.split("_")[1])
        by_year[year] += row["gain"]
    elif feat.startswith("change_"):
        change_total += row["gain"]

for y in YEARS:
    print(f"  AEF {y}: {by_year[y]:.0f}")
print(f"  All changes: {change_total:.0f}")

=== Step 1: Extract features (target = 389 per pixel) ===


train features:   0%|          | 0/16 [00:00<?, ?it/s]


Total labeled pixels: 14,636,136
Positives: 1,541,497 (10.53%)
Negatives: 13,094,639
Feature matrix: (14636136, 389), dtype=float32
Memory: 22.77 GB

=== Step 2: Subsample negatives 5x ===
After subsampling: 9,248,982 rows (16.7% positive)
Memory: 14.39 GB

Validation tile: 18NXH_6_8
  Train rows: 8,616,190, Val rows: 632,792

=== Step 4: Train LightGBM ===
Training until validation scores don't improve for 30 rounds
[50]	train's binary_logloss: 0.0710011	val's binary_logloss: 0.0970689
[100]	train's binary_logloss: 0.046931	val's binary_logloss: 0.0560598
[150]	train's binary_logloss: 0.0404155	val's binary_logloss: 0.0495472
[200]	train's binary_logloss: 0.0368198	val's binary_logloss: 0.0469071
[250]	train's binary_logloss: 0.0342635	val's binary_logloss: 0.0454907
[300]	train's binary_logloss: 0.0322989	val's binary_logloss: 0.0440329
[350]	train's binary_logloss: 0.0307287	val's binary_logloss: 0.043372
[400]	train's binary_logloss: 0.0294394	val's binary_logloss: 0.0429074
Did n

predicting:   0%|          | 0/5 [00:00<?, ?it/s]

  18NVJ_1_6: 976 positive pixels, 4 polygons
  18NYH_2_1: 104,886 positive pixels, 201 polygons
  33NTE_5_1: 38,495 positive pixels, 165 polygons
  47QMA_6_2: 1,198 positive pixels, 2 polygons
  48PWA_0_6: 79,018 positive pixels, 281 polygons

=== Submission saved: submission/submission.geojson ===
Total polygons: 653

=== Feature importance (top 15) ===
         feature         gain
change_2022_2023 9.220413e+06
change_2024_2025 5.788051e+06
     aef_2020_22 5.573185e+06
change_2023_2024 4.281284e+06
     aef_2020_44 2.331770e+06
change_2020_2021 2.055112e+06
change_2021_2022 1.999852e+06
      aef_2020_0 1.854208e+06
     aef_2020_36 9.613902e+05
      aef_2020_5 8.327484e+05
     aef_2025_33 7.140904e+05
      aef_2025_5 7.077656e+05
     aef_2020_60 7.001498e+05
     aef_2025_22 6.361402e+05
     aef_2020_51 6.262637e+05

=== Importance by year ===
  AEF 2020: 18896828
  AEF 2021: 1693667
  AEF 2022: 521119
  AEF 2023: 1113418
  AEF 2024: 1662949
  AEF 2025: 5075441
  All changes: 

In [9]:
!pip install lightgbm --break-system-packages



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
"""
Full pipeline: AEF + S2 monthly + S1 monthly + spatial context + deltas → LightGBM → GeoJSON.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from scipy.ndimage import uniform_filter
import lightgbm as lgb
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import sys, json

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt")
MODEL_DIR = Path("eda_artifacts/model"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR = Path("submission"); SUB_DIR.mkdir(parents=True, exist_ok=True)
TILE_PRED_DIR = Path("eda_artifacts/predictions"); TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64

# Per-year feature counts
N_AEF_PER_YEAR = 64          # 6 × 64 = 384
N_S2_PER_YEAR = 14           # mean, min, max, p10, p50, p90, iqr, final, drop, argmin_month per NDVI + mirror for NBR (10 NDVI + 4 NBR simplified)
N_S1_PER_YEAR = 4            # vv mean, std, min, range
N_CHANGES = len(YEARS) - 1   # 5 AEF year-pair cosine changes
N_SPATIAL_PER_CHANGE = 3     # 3x3 mean, 5x5 mean, 5x5 variance
N_SPATIAL = N_CHANGES * N_SPATIAL_PER_CHANGE  # 15
N_DELTAS_PER_PAIR = 3        # ndvi_yoy, nbr_yoy, vv_yoy
N_DELTAS = (len(YEARS) - 1) * N_DELTAS_PER_PAIR   # 15
N_REGION = 3                 # amazon/seasia/wafrica

N_FEATURES = (
    len(YEARS) * N_AEF_PER_YEAR +   # 384
    N_CHANGES +                      # 5
    len(YEARS) * N_S2_PER_YEAR +     # 84
    len(YEARS) * N_S1_PER_YEAR +     # 24
    N_SPATIAL +                      # 15
    N_DELTAS +                       # 15
    N_REGION                         # 3
)  # Total: 530

NEG_RATIO = 5
N_BOOST_ROUND = 1000
CV_HOLDOUT = "18NXH_6_8"


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best
    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape
    return None


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape
    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]
    return arr[:, row_idx[:, None], col_idx[None, :]]


# ============================================================
# AEF
# ============================================================
def load_aef_to_ref(tile, year, split, ref):
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"
    if not p.exists(): return None
    
    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)
        for b in range(N_AEF_BANDS):
            reproject(raw[b], aef[b],
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=ref[0], dst_crs=ref[1],
                      resampling=Resampling.bilinear)
    return aef


# ============================================================
# S2 MONTHLY → rich summary stats
# ============================================================
def load_s2_features(tile, year, split, ref):
    """Returns (14, H, W): NDVI (mean, min, range, p10, p50, p90, iqr, final, drop, argmin_month) + NBR (mean, min, range, final)."""
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    files = sorted(s2_dir.glob(f"*_{year}_*.tif"))
    
    ndvi_series, nbr_series = [], []
    for f in files:
        with rasterio.open(f) as r:
            if r.shape[0] < 100 or r.shape[1] < 100:
                continue
            try:
                bands = r.read([4, 8, 12]).astype(np.float32) / 10000.0  # Red, NIR, SWIR2
            except:
                continue
            n_bands = bands.shape[0]
            reproj = np.zeros((n_bands, *ref[2]), dtype=np.float32)
            for i in range(n_bands):
                reproject(bands[i], reproj[i],
                          src_transform=r.transform, src_crs=r.crs,
                          dst_transform=ref[0], dst_crs=ref[1],
                          resampling=Resampling.bilinear)
        
        red, nir, swir2 = reproj[0], reproj[1], reproj[2]
        valid = (red > 0) | (nir > 0)
        
        ndvi = (nir - red) / (nir + red + 1e-6)
        nbr = (nir - swir2) / (nir + swir2 + 1e-6)
        ndvi = np.where(valid, ndvi, np.nan)
        nbr = np.where(valid, nbr, np.nan)
        
        ndvi_series.append(ndvi)
        nbr_series.append(nbr)
    
    if len(ndvi_series) < 2:
        return None
    
    ndvi_stack = np.stack(ndvi_series)
    nbr_stack = np.stack(nbr_series)
    
    with np.errstate(all="ignore"):
        # NDVI rich stats
        ndvi_mean = np.nanmean(ndvi_stack, axis=0)
        ndvi_min = np.nanmin(ndvi_stack, axis=0)
        ndvi_max = np.nanmax(ndvi_stack, axis=0)
        ndvi_range = ndvi_max - ndvi_min
        ndvi_p10 = np.nanpercentile(ndvi_stack, 10, axis=0)
        ndvi_p50 = np.nanpercentile(ndvi_stack, 50, axis=0)
        ndvi_p90 = np.nanpercentile(ndvi_stack, 90, axis=0)
        ndvi_iqr = ndvi_p90 - ndvi_p10
        ndvi_final = ndvi_stack[-1]
        ndvi_drop = np.nanmin(np.diff(ndvi_stack, axis=0), axis=0) if len(ndvi_stack) > 1 else np.zeros_like(ndvi_mean)
        # Month of min NDVI (1-12)
        ndvi_argmin = np.nanargmin(np.where(np.isnan(ndvi_stack), np.inf, ndvi_stack), axis=0).astype(np.float32) + 1
        
        # NBR simplified stats
        nbr_mean = np.nanmean(nbr_stack, axis=0)
        nbr_min = np.nanmin(nbr_stack, axis=0)
        nbr_range = np.nanmax(nbr_stack, axis=0) - nbr_min
        nbr_final = nbr_stack[-1]
    
    out = np.stack([
        ndvi_mean, ndvi_min, ndvi_range, ndvi_p10, ndvi_p50, ndvi_p90,
        ndvi_iqr, ndvi_final, ndvi_drop, ndvi_argmin,
        nbr_mean, nbr_min, nbr_range, nbr_final
    ])
    return np.nan_to_num(out, nan=0.0).astype(np.float32)


# ============================================================
# S1 MONTHLY → summary stats
# ============================================================
def load_s1_features(tile, year, split, ref):
    s1_dir = ROOT / f"sentinel-1/{split}/{tile}__s1_rtc"
    files = sorted(s1_dir.glob(f"*_{year}_*.tif"))
    
    vv_series = []
    for f in files:
        with rasterio.open(f) as r:
            if r.shape[0] < 100 or r.shape[1] < 100:
                continue
            try:
                vv = r.read(1).astype(np.float32)
            except:
                continue
            out = np.zeros(ref[2], dtype=np.float32)
            reproject(vv, out,
                      src_transform=r.transform, src_crs=r.crs,
                      dst_transform=ref[0], dst_crs=ref[1],
                      resampling=Resampling.bilinear)
        out = np.where(out > 0, out, np.nan)
        vv_series.append(out)
    
    if len(vv_series) < 2:
        return None
    
    vv_stack = np.stack(vv_series)
    with np.errstate(all="ignore"):
        vv_mean = np.nanmean(vv_stack, axis=0)
        vv_std = np.nanstd(vv_stack, axis=0)
        vv_min = np.nanmin(vv_stack, axis=0)
        vv_range = np.nanmax(vv_stack, axis=0) - vv_min
    
    out = np.stack([vv_mean, vv_std, vv_min, vv_range])
    return np.nan_to_num(out, nan=0.0).astype(np.float32)


# ============================================================
# SPATIAL CONTEXT FEATURES (on AEF change)
# ============================================================
def compute_spatial_features(change):
    """Per AEF change map, compute: 3x3 mean, 5x5 mean, 5x5 variance."""
    mean_3x3 = uniform_filter(change, size=3).astype(np.float32)
    mean_5x5 = uniform_filter(change, size=5).astype(np.float32)
    # Variance = E[X^2] - (E[X])^2
    sq_mean_5x5 = uniform_filter(change**2, size=5)
    var_5x5 = (sq_mean_5x5 - mean_5x5**2).astype(np.float32)
    return np.stack([mean_3x3, mean_5x5, var_5x5])


# ============================================================
# REGION ENCODING
# ============================================================
def region_features(tile, shape):
    """Return (3, H, W) one-hot: [amazon, seasia, wafrica]."""
    zone = tile[:2]
    amazon = 1.0 if zone in ("18", "19") else 0.0
    seasia = 1.0 if zone in ("47", "48") else 0.0
    wafrica = 1.0 if zone == "33" else 0.0
    
    h, w = shape
    return np.stack([
        np.full(shape, amazon, dtype=np.float32),
        np.full(shape, seasia, dtype=np.float32),
        np.full(shape, wafrica, dtype=np.float32),
    ])


# ============================================================
# FEATURE EXTRACTION
# ============================================================
def extract_features(tile, split):
    ref = get_ref(tile, split)
    if ref is None: return None, None
    
    # AEF per year
    aef_stack = []
    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)
        if aef is None: return None, None
        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)
    
    # AEF changes
    changes = []
    for i in range(len(YEARS) - 1):
        a, b = aef_stack[i], aef_stack[i + 1]
        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b
        cos = np.where(denom > 1e-6, dot / (denom + 1e-9), 0.0).astype(np.float32)
        changes.append(cos)
    
    # Spatial context on each change map
    spatial_feats = []
    for ch in changes:
        spatial_feats.append(compute_spatial_features(ch))
    # Shape: list of (3, H, W) → concatenate to (15, H, W)
    spatial_all = np.concatenate(spatial_feats, axis=0)
    
    # S2 per year
    s2_stack = []
    for year in YEARS:
        s2 = load_s2_features(tile, year, split, ref)
        if s2 is None:
            s2 = np.zeros((N_S2_PER_YEAR, *ref[2]), dtype=np.float32)
        s2 = resize_nearest(s2, TARGET_SHAPE)
        s2_stack.append(s2)
    
    # S1 per year
    s1_stack = []
    for year in YEARS:
        s1 = load_s1_features(tile, year, split, ref)
        if s1 is None:
            s1 = np.zeros((N_S1_PER_YEAR, *ref[2]), dtype=np.float32)
        s1 = resize_nearest(s1, TARGET_SHAPE)
        s1_stack.append(s1)
    
    # Year-to-year deltas: NDVI final, NBR final, VV mean
    # S2 stats index: 7 = ndvi_final, 13 = nbr_final
    # S1 stats index: 0 = vv_mean
    deltas = []
    for i in range(len(YEARS) - 1):
        ndvi_delta = s2_stack[i+1][7] - s2_stack[i][7]
        nbr_delta = s2_stack[i+1][13] - s2_stack[i][13]
        vv_delta = s1_stack[i+1][0] - s1_stack[i][0]
        deltas.extend([ndvi_delta, nbr_delta, vv_delta])
    deltas_all = np.stack(deltas).astype(np.float32)
    
    # Region encoding
    region_feats = region_features(tile, TARGET_SHAPE)
    
    # Stack everything
    all_feats = np.concatenate(
        aef_stack +                                   # 384
        [c[None] for c in changes] +                  # 5
        s2_stack +                                    # 84
        s1_stack +                                    # 24
        [spatial_all] +                               # 15
        [deltas_all] +                                # 15
        [region_feats],                               # 3
        axis=0
    )  # (530, H, W)
    features = all_feats.reshape(N_FEATURES, -1).T
    return features, ref


def feature_names():
    names = []
    # AEF
    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")
    # AEF changes
    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i+1]}")
    # S2
    for y in YEARS:
        for stat in ["ndvi_mean", "ndvi_min", "ndvi_range", "ndvi_p10", "ndvi_p50",
                     "ndvi_p90", "ndvi_iqr", "ndvi_final", "ndvi_drop", "ndvi_argmin",
                     "nbr_mean", "nbr_min", "nbr_range", "nbr_final"]:
            names.append(f"s2_{y}_{stat}")
    # S1
    for y in YEARS:
        for stat in ["vv_mean", "vv_std", "vv_min", "vv_range"]:
            names.append(f"s1_{y}_{stat}")
    # Spatial
    for i in range(len(YEARS) - 1):
        for stat in ["3x3_mean", "5x5_mean", "5x5_var"]:
            names.append(f"spatial_change_{YEARS[i]}_{YEARS[i+1]}_{stat}")
    # Deltas
    for i in range(len(YEARS) - 1):
        for stat in ["ndvi_yoy", "nbr_yoy", "vv_yoy"]:
            names.append(f"delta_{YEARS[i]}_{YEARS[i+1]}_{stat}")
    # Region
    names.extend(["region_amazon", "region_seasia", "region_wafrica"])
    return names


# ============================================================
# STEP 1 — Extract training features
# ============================================================
print(f"=== Extracting features: {N_FEATURES} per pixel ===")
train_tiles = sorted([p.name.replace("__s2_l2a", "")
                      for p in (ROOT / "sentinel-2/train").iterdir()])

all_feats, all_labels, all_weights, all_tiles = [], [], [], []
for tile in tqdm(train_tiles, desc="train features"):
    pgt_path = PGT_DIR / f"{tile}.npz"
    if not pgt_path.exists(): continue
    
    feats, ref = extract_features(tile, "train")
    if feats is None: continue
    
    pgt = np.load(pgt_path)
    label = pgt["label"].flatten()
    conf = pgt["confidence"].flatten()
    
    valid = ~np.isnan(label)
    all_feats.append(feats[valid])
    all_labels.append(label[valid])
    all_weights.append(conf[valid])
    all_tiles.append(np.full(int(valid.sum()), tile))

X = np.concatenate(all_feats, axis=0)
y = np.concatenate(all_labels).astype(np.float32)
w = np.concatenate(all_weights).astype(np.float32)
tile_ids = np.concatenate(all_tiles)
del all_feats, all_labels, all_weights, all_tiles

print(f"\nTotal labeled pixels: {len(y):,}")
print(f"Positives: {int((y == 1).sum()):,} ({(y == 1).mean():.2%})")
print(f"Feature matrix: {X.shape}, mem: {X.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 2 — Subsample negatives
# ============================================================
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_RATIO)
rng = np.random.default_rng(42)
neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)
keep = np.concatenate([pos_idx, neg_sampled])
rng.shuffle(keep)

X_kept = X[keep]; y_kept = y[keep]; w_kept = w[keep]; tiles_kept = tile_ids[keep]
del X, y, w, tile_ids

print(f"After subsample: {len(y_kept):,} rows, mem: {X_kept.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 3 — Train/val split
# ============================================================
val_mask = tiles_kept == CV_HOLDOUT
X_tr = X_kept[~val_mask]; y_tr = y_kept[~val_mask]; w_tr = w_kept[~val_mask]
X_val = X_kept[val_mask]; y_val = y_kept[val_mask]

print(f"Train: {len(y_tr):,}, Val ({CV_HOLDOUT}): {len(y_val):,}")


# ============================================================
# STEP 4 — Train LightGBM
# ============================================================
print("\n=== Training LightGBM ===")
names = feature_names()
assert len(names) == N_FEATURES, f"Mismatch: {len(names)} vs {N_FEATURES}"

lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr, feature_name=names)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.02,
    "num_leaves": 255,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 5,
    "lambda_l2": 1.0,
    "verbose": -1,
    "n_jobs": -1,
}

model = lgb.train(
    params, lgb_train,
    num_boost_round=N_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)

model.save_model(str(MODEL_DIR / "lgbm_full.txt"))


# ============================================================
# STEP 5 — Tune threshold
# ============================================================
val_pred = model.predict(X_val)
best_thr, best_f1 = 0.5, 0
for thr in np.linspace(0.05, 0.95, 37):
    f1 = f1_score(y_val, val_pred > thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr
print(f"\nBest threshold: {best_thr:.3f} (F1={best_f1:.3f})")


# ============================================================
# STEP 6 — Predict test tiles
# ============================================================
print("\n=== Predicting test tiles ===")
test_tiles = sorted([p.name.replace("__s2_l2a", "")
                     for p in (ROOT / "sentinel-2/test").iterdir()])

all_features_geojson = []

for tile in tqdm(test_tiles, desc="test"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped"); continue
    
    pred = model.predict(feats)
    binary = (pred > best_thr).astype(np.uint8).reshape(TARGET_SHAPE)
    
    # Resize to native resolution
    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE
    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)
    binary_native = binary[row_idx[:, None], col_idx[None, :]]
    
    tile_path = TILE_PRED_DIR / f"{tile}_pred.tif"
    profile = {
        "driver": "GTiff", "height": h_native, "width": w_native,
        "count": 1, "dtype": "uint8",
        "crs": ref[1], "transform": ref[0], "nodata": 0,
    }
    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)
    
    geojson = raster_to_geojson(str(tile_path), output_path=None)
    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile
    all_features_geojson.extend(geojson["features"])
    print(f"  {tile}: {int(binary_native.sum()):,} px, {len(geojson['features'])} polygons")


# ============================================================
# STEP 7 — Save submission
# ============================================================
submission = {"type": "FeatureCollection", "features": all_features_geojson}
sub_path = SUB_DIR / "submission.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)
print(f"\n=== Saved: {sub_path}, {len(all_features_geojson)} polygons ===")


# ============================================================
# STEP 8 — Feature importance
# ============================================================
importance = pd.DataFrame({
    "feature": names,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print(f"\nTop 20 features:")
print(importance.head(20).to_string(index=False))

print(f"\nGain by modality:")
modality_gain = {"aef_": 0, "change_": 0, "s2_": 0, "s1_": 0, "spatial_": 0, "delta_": 0, "region_": 0}
for _, row in importance.iterrows():
    for mod in modality_gain:
        if row["feature"].startswith(mod):
            modality_gain[mod] += row["gain"]
            break
for mod, g in sorted(modality_gain.items(), key=lambda x: -x[1]):
    print(f"  {mod}: {g:.0f}")

=== Extracting features: 530 per pixel ===


train features:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_5944/275136043.py:153: RuntimeWarning: Mean of empty slice
  ndvi_mean = np.nanmean(ndvi_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:154: RuntimeWarning: All-NaN slice encountered
  ndvi_min = np.nanmin(ndvi_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:155: RuntimeWarning: All-NaN slice encountered
  ndvi_max = np.nanmax(ndvi_stack, axis=0)
/opt/venv/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/tmp/ipykernel_5944/275136043.py:162: RuntimeWarning: All-NaN slice encountered
  ndvi_drop = np.nanmin(np.diff(ndvi_stack, axis=0), axis=0) if len(ndvi_stack) > 1 else np.zeros_like(ndvi_mean)
/tmp/ipykernel_5944/275136043.py:167: RuntimeWarning: Mean of empty slice
  nbr_mean = np.nanmean(nbr_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:168: RuntimeWarning: All-NaN slice encountered
  nbr_min = np.nanmin(nbr_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:169: RuntimeWarning: A


Total labeled pixels: 14,636,136
Positives: 1,541,497 (10.53%)
Feature matrix: (14636136, 530), mem: 31.03 GB
After subsample: 9,248,982 rows, mem: 19.61 GB
Train: 8,616,190, Val (18NXH_6_8): 632,792

=== Training LightGBM ===
Training until validation scores don't improve for 50 rounds
[50]	train's binary_logloss: 0.132825	val's binary_logloss: 0.287217
[100]	train's binary_logloss: 0.0721932	val's binary_logloss: 0.150424
[150]	train's binary_logloss: 0.0498986	val's binary_logloss: 0.100915
[200]	train's binary_logloss: 0.0399994	val's binary_logloss: 0.0787357
[250]	train's binary_logloss: 0.0347154	val's binary_logloss: 0.0692397
[300]	train's binary_logloss: 0.0313094	val's binary_logloss: 0.0646281
[350]	train's binary_logloss: 0.0288532	val's binary_logloss: 0.0622026
[400]	train's binary_logloss: 0.0268329	val's binary_logloss: 0.0601702
[450]	train's binary_logloss: 0.0251728	val's binary_logloss: 0.0584771
[500]	train's binary_logloss: 0.0236983	val's binary_logloss: 0.0577

test:   0%|          | 0/5 [00:00<?, ?it/s]

...

  33NTE_5_1: 55,767 px, 192 polygons


/tmp/ipykernel_5944/275136043.py:153: RuntimeWarning: Mean of empty slice
  ndvi_mean = np.nanmean(ndvi_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:154: RuntimeWarning: All-NaN slice encountered
  ndvi_min = np.nanmin(ndvi_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:155: RuntimeWarning: All-NaN slice encountered
  ndvi_max = np.nanmax(ndvi_stack, axis=0)
/opt/venv/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/tmp/ipykernel_5944/275136043.py:162: RuntimeWarning: All-NaN slice encountered
  ndvi_drop = np.nanmin(np.diff(ndvi_stack, axis=0), axis=0) if len(ndvi_stack) > 1 else np.zeros_like(ndvi_mean)
/tmp/ipykernel_5944/275136043.py:167: RuntimeWarning: Mean of empty slice
  nbr_mean = np.nanmean(nbr_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:168: RuntimeWarning: All-NaN slice encountered
  nbr_min = np.nanmin(nbr_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:169: RuntimeWarning: A

  47QMA_6_2: 2,665 px, 11 polygons


/tmp/ipykernel_5944/275136043.py:153: RuntimeWarning: Mean of empty slice
  ndvi_mean = np.nanmean(ndvi_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:154: RuntimeWarning: All-NaN slice encountered
  ndvi_min = np.nanmin(ndvi_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:155: RuntimeWarning: All-NaN slice encountered
  ndvi_max = np.nanmax(ndvi_stack, axis=0)
/opt/venv/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/tmp/ipykernel_5944/275136043.py:162: RuntimeWarning: All-NaN slice encountered
  ndvi_drop = np.nanmin(np.diff(ndvi_stack, axis=0), axis=0) if len(ndvi_stack) > 1 else np.zeros_like(ndvi_mean)
/tmp/ipykernel_5944/275136043.py:167: RuntimeWarning: Mean of empty slice
  nbr_mean = np.nanmean(nbr_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:168: RuntimeWarning: All-NaN slice encountered
  nbr_min = np.nanmin(nbr_stack, axis=0)
/tmp/ipykernel_5944/275136043.py:169: RuntimeWarning: A

  48PWA_0_6: 84,758 px, 282 polygons

=== Saved: submission/submission.geojson, 685 polygons ===

Top 20 features:
                          feature         gain
                      aef_2020_22 1.291676e+07
spatial_change_2022_2023_5x5_mean 1.190842e+07
                 change_2022_2023 9.408615e+06
spatial_change_2024_2025_5x5_mean 7.572741e+06
spatial_change_2024_2025_3x3_mean 6.980989e+06
spatial_change_2022_2023_3x3_mean 5.961366e+06
spatial_change_2023_2024_3x3_mean 3.727310e+06
                 change_2024_2025 2.913825e+06
                 change_2021_2022 2.860237e+06
spatial_change_2020_2021_5x5_mean 2.669200e+06
                       aef_2020_0 2.611955e+06
                      aef_2020_44 2.525188e+06
                 change_2020_2021 2.048235e+06
spatial_change_2020_2021_3x3_mean 1.945646e+06
                      aef_2020_36 1.846980e+06
                       aef_2020_3 1.475912e+06
                      aef_2020_51 1.453670e+06
                       aef_2020_5 1.107

In [ ]:
print('hi')

In [17]:
"""
Extract test features ONE TILE AT A TIME with progress visibility.
Saves each tile immediately — resumable if interrupted.
"""

from pathlib import Path
import numpy as np
import gc
import time
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

FEATURES_DIR = Path("eda_artifacts/features_full")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# All 5 test tiles
test_tiles = sorted([p.name.replace("__s2_l2a", "")
                     for p in (ROOT / "sentinel-2/test").iterdir()])

print(f"Extracting test features for {len(test_tiles)} tiles")
print(f"Output: {FEATURES_DIR}/\n")
print("=" * 70)

overall_start = time.time()

for i, tile in enumerate(test_tiles, 1):
    out_path = FEATURES_DIR / f"test_{tile}.npz"
    
    # Skip if already done (resumable)
    if out_path.exists():
        file_size_mb = out_path.stat().st_size / 1e6
        print(f"[{i}/{len(test_tiles)}] {tile}: ALREADY SAVED ({file_size_mb:.1f} MB) — skipping")
        continue
    
    print(f"\n[{i}/{len(test_tiles)}] {tile}: extracting...")
    tile_start = time.time()
    
    try:
        feats, ref = extract_features(tile, "test")
        
        if feats is None:
            print(f"  SKIPPED (no features)")
            continue
        
        extract_time = time.time() - tile_start
        print(f"  Extracted in {extract_time:.1f}s: shape {feats.shape}, {feats.nbytes / 1e9:.2f} GB")
        
        # Save with all metadata needed to recover ref later
        save_start = time.time()
        np.savez_compressed(
            out_path,
            X=feats.astype(np.float32),
            transform=np.array(list(ref[0]), dtype=np.float64),
            crs_wkt=np.array(ref[1].to_wkt() if ref[1] else "", dtype=object),
            shape=np.array(list(ref[2]), dtype=np.int64),
        )
        save_time = time.time() - save_start
        file_size_mb = out_path.stat().st_size / 1e6
        
        print(f"  Saved in {save_time:.1f}s: {file_size_mb:.1f} MB compressed")
        print(f"  Total for this tile: {time.time() - tile_start:.1f}s")
        
        # Free memory
        del feats, ref
        gc.collect()
    
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")
        # Continue with next tile
        continue
    
    # Progress summary
    elapsed = time.time() - overall_start
    tiles_done = i
    avg_per_tile = elapsed / tiles_done
    remaining = (len(test_tiles) - tiles_done) * avg_per_tile
    print(f"  Progress: {tiles_done}/{len(test_tiles)} done, "
          f"{elapsed/60:.1f}m elapsed, ~{remaining/60:.1f}m remaining")

print("\n" + "=" * 70)
print(f"DONE. Total time: {(time.time() - overall_start) / 60:.1f} min")

# Summary of saved files
print(f"\nSaved files in {FEATURES_DIR}/:")
total_size = 0
for p in sorted(FEATURES_DIR.glob("test_*.npz")):
    size_mb = p.stat().st_size / 1e6
    total_size += size_mb
    print(f"  {p.name}: {size_mb:.1f} MB")
print(f"Total: {total_size:.1f} MB")

Extracting test features for 5 tiles
Output: eda_artifacts/features_full/


[1/5] 18NVJ_1_6: extracting...


KeyboardInterrupt: 

In [30]:
"""
Re-run test predictions with threshold 0.12.
Test features keyed by tile name in single combined npz.
"""

from pathlib import Path
import numpy as np
import rasterio
from affine import Affine
from rasterio.crs import CRS
import lightgbm as lgb
from tqdm.auto import tqdm
import sys, json, warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
from submission_utils import raster_to_geojson

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
MODEL_DIR = Path("eda_artifacts/model")
MODEL_PATH = MODEL_DIR / "lgbm_full.txt"
FEATURES_DIR = Path("eda_artifacts/features_full")
SUB_DIR = Path("submission"); SUB_DIR.mkdir(exist_ok=True)
TILE_PRED_DIR = Path("eda_artifacts/predictions_thr12"); TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)
PROB_DIR = Path("eda_artifacts/probabilities"); PROB_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_PATH = FEATURES_DIR / "test_features.npz"
TARGET_SHAPE = (1000, 1000)
N_FEATURES = 530
THRESHOLD = 0.08


# ============================================================
# GET REF from satellite data (needed for georeferencing)
# ============================================================
def get_ref(tile):
    s2_dir = ROOT / f"sentinel-2/test/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None: return best
    aef_files = sorted((ROOT / "aef-embeddings/test").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape
    return None


# ============================================================
# LOAD MODEL
# ============================================================
print(f"Loading model: {MODEL_PATH}")
model = lgb.Booster(model_file=str(MODEL_PATH))
assert model.num_feature() == N_FEATURES
print(f"Threshold: {THRESHOLD}\n")


# ============================================================
# LOAD COMBINED TEST FEATURES
# ============================================================
print(f"Loading {COMBINED_PATH} (lazy)...")
data = np.load(COMBINED_PATH, allow_pickle=True)
print(f"Tiles in file: {list(data.files)}\n")


# ============================================================
# PREDICT EACH TILE
# ============================================================
all_features_geojson = []

for tile in tqdm(data.files, desc="predict"):
    print(f"\n{'='*60}")
    print(f"Tile: {tile}")
    print("="*60)
    
    # Get ref for this tile
    ref = get_ref(tile)
    if ref is None:
        print(f"  SKIP: no ref found")
        continue
    print(f"  ref: shape {ref[2]}, crs {ref[1]}")
    
    # Load features (this is the slow part — loading 2GB from compressed npz)
    print(f"  loading features...")
    feats = data[tile]
    print(f"  features: {feats.shape}, {feats.nbytes / 1e9:.2f} GB")
    
    # Predict
    print(f"  predicting...")
    prob = model.predict(feats).reshape(TARGET_SHAPE).astype(np.float32)
    np.save(PROB_DIR / f"lgb_{tile}.npy", prob)
    
    binary = (prob > THRESHOLD).astype(np.uint8)
    
    print(f"  prob range: [{prob.min():.3f}, {prob.max():.3f}], mean {prob.mean():.3f}")
    print(f"  positives at thr={THRESHOLD}: {int(binary.sum()):,} ({binary.mean():.2%} of tile)")
    
    # Resize to native resolution
    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE
    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)
    binary_native = binary[row_idx[:, None], col_idx[None, :]]
    
    if binary_native.sum() == 0:
        print(f"  0 positives after resize, skipping")
        del feats, prob, binary
        continue
    
    # Save GeoTIFF
    tile_path = TILE_PRED_DIR / f"{tile}_pred.tif"
    profile = {"driver": "GTiff", "height": h_native, "width": w_native,
               "count": 1, "dtype": "uint8", "crs": ref[1],
               "transform": ref[0], "nodata": 0}
    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)
    
    # Convert to polygons
    try:
        geojson = raster_to_geojson(str(tile_path), output_path=None)
        for feat in geojson["features"]:
            feat["properties"]["tile"] = tile
        all_features_geojson.extend(geojson["features"])
        print(f"  saved: {len(geojson['features'])} polygons")
    except ValueError as e:
        print(f"  No polygons: {e}")
    
    # Free memory
    del feats, prob, binary, binary_native
    import gc; gc.collect()


# ============================================================
# SAVE SUBMISSION
# ============================================================
submission = {"type": "FeatureCollection", "features": all_features_geojson}
sub_path = SUB_DIR / "submission_full_thr12.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\n{'='*60}")
print(f"SAVED: {sub_path}")
print("="*60)
print(f"Total polygons: {len(all_features_geojson)}")
print(f"Probability maps: {PROB_DIR}/")
print(f"Threshold used: {THRESHOLD}")

Loading model: eda_artifacts/model/lgbm_full.txt
Threshold: 0.08

Loading eda_artifacts/features_full/test_features.npz (lazy)...
Tiles in file: ['18NVJ_1_6', '18NYH_2_1', '33NTE_5_1', '47QMA_6_2', '48PWA_0_6']



predict:   0%|          | 0/5 [00:00<?, ?it/s]


Tile: 18NVJ_1_6
  ref: shape (1002, 1002), crs EPSG:32618
  loading features...
  features: (1000000, 530), 2.12 GB
  predicting...
  prob range: [0.000, 0.905], mean 0.002
  positives at thr=0.08: 1,714 (0.17% of tile)
  saved: 8 polygons

Tile: 18NYH_2_1
  ref: shape (1004, 1004), crs EPSG:32618
  loading features...
  features: (1000000, 530), 2.12 GB
  predicting...
  prob range: [0.000, 1.000], mean 0.084
  positives at thr=0.08: 123,292 (12.33% of tile)
  saved: 205 polygons

Tile: 33NTE_5_1
  ref: shape (1006, 1007), crs EPSG:32633
  loading features...
  features: (1000000, 530), 2.12 GB
  predicting...
  prob range: [0.000, 0.997], mean 0.039
  positives at thr=0.08: 67,577 (6.76% of tile)
  saved: 202 polygons

Tile: 47QMA_6_2


RasterioIOError: 'data/makeathon-challenge/sentinel-2/test/47QMA_6_2__s2_l2a/47QMA_6_2__s2_l2a_2020_4.tif' not recognized as being in a supported file format.

In [26]:
import numpy as np
from pathlib import Path

p = Path("eda_artifacts/features_full/test_features.npz")
print(f"File size: {p.stat().st_size / 1e9:.2f} GB\n")

data = np.load(p, allow_pickle=True)
print(f"Keys in file: {list(data.files)}\n")

# Show shape + type of each key
for key in data.files:
    arr = data[key]
    print(f"  '{key}': shape {arr.shape}, dtype {arr.dtype}")

File size: 8.99 GB

Keys in file: ['18NVJ_1_6', '18NYH_2_1', '33NTE_5_1', '47QMA_6_2', '48PWA_0_6']

  '18NVJ_1_6': shape (1000000, 530), dtype float32
  '18NYH_2_1': shape (1000000, 530), dtype float32
  '33NTE_5_1': shape (1000000, 530), dtype float32
  '47QMA_6_2': shape (1000000, 530), dtype float32
  '48PWA_0_6': shape (1000000, 530), dtype float32
